## 1. Import và tiện ích kiểm thử
Mã `diagonalize` nằm trong `diagonalization.py` (Python thuần, không NumPy). Notebook này chỉ import và định nghĩa các hàm nhỏ để nhân ma trận, norm Frobenius và sai số tương đối khi kiểm tra \(A \approx P D P^{-1}\).

Chạy notebook với thư mục làm việc là thư mục chứa `diagonalization.py`.

In [4]:
import os
import sys
from pathlib import Path

REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)

_cwd = os.getcwd()
if os.path.isfile(os.path.join(_cwd, "diagonalization.py")):
    if _cwd not in sys.path:
        sys.path.insert(0, _cwd)
else:
    raise ImportError(
        "Không thấy diagonalization.py trong thư mục làm việc hiện tại. "
        f"CWD={_cwd!r}. Mở đúng folder chứa diagonalization.py hoặc os.chdir(...)."
    )

from diagonalization import diagonalize


def matmul(A, B):
    rows = len(A)
    cols = len(B[0])
    inner = len(B)
    result = [[0 for _ in range(cols)] for _ in range(rows)]
    for i in range(rows):
        for j in range(cols):
            s = 0
            for k in range(inner):
                s += A[i][k] * B[k][j]
            result[i][j] = s
    return result


def frobenius_norm(M):
    s = 0.0
    for row in M:
        for x in row:
            s += abs(x) ** 2
    return s ** 0.5


def relative_error(A_ref, A_test):
    diff = []
    for i in range(len(A_ref)):
        row = []
        for j in range(len(A_ref[0])):
            row.append(A_test[i][j] - A_ref[i][j])
        diff.append(row)
    return frobenius_norm(diff) / max(frobenius_norm(A_ref), 1e-30)


def reconstruction_rel_err(A, tol=1e-10):
    """Trả về sai số tương đối Frobenius của A - P D P^{-1}; gây lỗi nếu vượt tol."""
    P, D, P_inv = diagonalize(A)
    recon = matmul(matmul(P, D), P_inv)
    err = relative_error(A, recon)
    if err >= tol:
        raise AssertionError(f"Sai số tương đối {err:.3e} >= {tol:.3e}")
    return err, P, D, P_inv

## 2. Kiểm thử tái tạo \(A = P D P^{-1}\) (5 trường hợp)
Ma trận tam giác trên tổng quát, đối xứng, đường chéo, phổ thực (2×2), và tam giác trên với trị riêng đã biết (3 và 7). Sai số tương đối cần \(< 10^{-10}\).

In [5]:
from diagonalization import diagonalize

def matmul(A, B):
    rows = len(A)
    cols = len(B[0])
    inner = len(B)
    result = [[0 for _ in range(cols)] for _ in range(rows)]
    for i in range(rows):
        for j in range(cols):
            s = 0
            for k in range(inner):
                s += A[i][k] * B[k][j]
            result[i][j] = s
    return result


def frobenius_norm(M):
    s = 0.0
    for row in M:
        for x in row:
            s += abs(x) ** 2
    return s ** 0.5


def relative_error(A_ref, A_test):
    diff = []
    for i in range(len(A_ref)):
        row = []
        for j in range(len(A_ref[0])):
            row.append(A_test[i][j] - A_ref[i][j])
        diff.append(row)
    return frobenius_norm(diff) / max(frobenius_norm(A_ref), 1e-30)


def reconstruction_rel_err(A, tol=1e-10):
    P, D, P_inv = diagonalize(A)
    recon = matmul(matmul(P, D), P_inv)
    err = relative_error(A, recon)
    if err >= tol:
        raise AssertionError(f"Sai số tương đối {err:.3e} >= {tol:.3e}")
    return err, P, D, P_inv


def test_diagonalize_reconstruction():
    print("--- Testing diagonalize reconstruction (A ~ P D P^-1) ---")

    # Case 1: 2×2 tam giác trên tổng quát
    A1 = [[4, 1], [0, 2]]
    e1, _, _, _ = reconstruction_rel_err(A1)
    print(f"1. Upper-triangular 2x2: rel_err={e1:.3e} (expected < 1e-10)")

    # Case 2: Đối xứng
    A2 = [[2, 1], [1, 2]]
    e2, _, _, _ = reconstruction_rel_err(A2)
    print(f"2. Symmetric: rel_err={e2:.3e} (expected < 1e-10)")

    # Case 3: Đường chéo — trị riêng 5 và 3
    A3 = [[5, 0], [0, 3]]
    e3, _, D3, _ = reconstruction_rel_err(A3)
    vals3 = sorted([D3[0][0], D3[1][1]])
    print(
        f"3. Diagonal: rel_err={e3:.3e}, eig(sorted)={vals3} (expected [3.0, 5.0])"
    )

    # Case 4: Phổ thực (2×2)
    A4 = [[3, 1], [0, 1]]
    e4, _, D4, _ = reconstruction_rel_err(A4)
    print(
        f"4. Real spectrum 2x2: rel_err={e4:.3e}, D[0][0]={D4[0][0]}, D[1][1]={D4[1][1]}"
    )

    # Case 5: Tam giác trên — trị riêng đường chéo 7 và 3
    A5 = [[7, 2], [0, 3]]
    e5, _, D5, _ = reconstruction_rel_err(A5)
    got5 = sorted([D5[0][0], D5[1][1]])
    print(f"5. Upper triangular: rel_err={e5:.3e}, eig(sorted)={got5} (expected [3.0, 7.0])")

    # Case 6: 5×5 có trị riêng lặp lại để che phủ đường dẫn n×n và deflation/eigenspace assembly
    P5 = [
        [1, 0, 0, 0, 0],
        [1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1],
    ]
    P5_inv = [
        [1, 0, 0, 0, 0],
        [-1, 1, 0, 0, 0],
        [0, -1, 1, 0, 0],
        [0, 0, -1, 1, 0],
        [0, 0, 0, -1, 1],
    ]
    eigenvalues5 = [2, 2, 3, 3, 5]
    D5 = [[eigenvalues5[i] if i == j else 0 for j in range(5)] for i in range(5)]
    A5_big = matmul(matmul(P5, D5), P5_inv)
    e5_big, _, D5_big, _ = reconstruction_rel_err(A5_big)
    eig5_big = sorted([D5_big[i][i] for i in range(5)])
    expected5 = [float(x) for x in sorted(eigenvalues5)]
    assert all(abs(a - b) < 1e-8 for a, b in zip(eig5_big, expected5)), (
        f"6. 5x5 repeated eigenvalues: expected {expected5}, got {eig5_big}"
    )
    print(
        f"6. 5x5 repeated eigenvalues: rel_err={e5_big:.3e}, eig(sorted)={eig5_big}"
    )

    # Case 7: Diagonal 4x4
    A7 = [
        [1, 0, 0, 0],
        [0, 2, 0, 0],
        [0, 0, 3, 0],
        [0, 0, 0, 4],
    ]
    e7, _, D7, _ = reconstruction_rel_err(A7)
    eig7 = sorted([D7[i][i] for i in range(4)])
    print(f"7. Diagonal 4x4: rel_err={e7:.3e}, eig={eig7}")

    # Case 8: Symmetric 3x3
    A8 = [
        [4, 1, 1],
        [1, 3, 0],
        [1, 0, 2],
    ]
    e8, _, _, _ = reconstruction_rel_err(A8)
    print(f"8. Symmetric 3x3: rel_err={e8:.3e}")

    # Case 9: Upper triangular 4x4
    A9 = [
        [5, 2, 1, 0],
        [0, 4, 3, 1],
        [0, 0, 3, 2],
        [0, 0, 0, 1],
    ]
    e9, _, D9, _ = reconstruction_rel_err(A9)
    eig9 = sorted([D9[i][i] for i in range(4)])
    print(f"9. Upper triangular 4x4: rel_err={e9:.3e}, eig={eig9}")


test_diagonalize_reconstruction()

--- Testing diagonalize reconstruction (A ~ P D P^-1) ---
1. Upper-triangular 2x2: rel_err=0.000e+00 (expected < 1e-10)
2. Symmetric: rel_err=4.441e-16 (expected < 1e-10)
3. Diagonal: rel_err=0.000e+00, eig(sorted)=[3, 5] (expected [3.0, 5.0])
4. Real spectrum 2x2: rel_err=0.000e+00, D[0][0]=1, D[1][1]=3
5. Upper triangular: rel_err=0.000e+00, eig(sorted)=[3, 7] (expected [3.0, 7.0])
6. 5x5 repeated eigenvalues: rel_err=4.562e-14, eig(sorted)=[1.999999999999801, 2.0, 3.000000000000022, 3.000000000000177, 5.0]
7. Diagonal 4x4: rel_err=0.000e+00, eig=[1, 2, 3, 4]
8. Symmetric 3x3: rel_err=3.207e-12
9. Upper triangular 4x4: rel_err=2.043e-15, eig=[1, 3, 4, 5]


## 3. Kiểm thử xử lý lỗi (3 trường hợp)
Ma trận không vuông, không chéo hoá được (khối Jordan 2×2), và ma trận có trị riêng phức (xoay 90°) — cần báo lỗi rõ ràng.

In [6]:
def test_diagonalize_errors():
    print("--- Testing diagonalize error handling ---")

    # Case 1: Không vuông
    try:
        diagonalize([[1, 2, 3], [4, 5, 6]])
        raise AssertionError("1. Non-square: expected ValueError")
    except ValueError as ex:
        assert "square" in str(ex).lower(), f"1. Non-square: unexpected message: {ex}"
        print(f"1. Non-square: OK — ValueError: {ex}")

    # Case 2: Không chéo hoá được (Jordan block)
    try:
        diagonalize([[1, 1], [0, 1]])
        raise AssertionError("2. Not diagonalizable: expected ValueError")
    except ValueError as ex:
        ok = "not diagonalizable" in str(ex).lower()
        assert ok, f"2. Not diagonalizable: unexpected message: {ex}"
        tag = "OK" if ok else "OK (unexpected message)"
        print(f"2. Not diagonalizable: {tag} — {ex}")

    # Case 3: Trị riêng phức (2×2)
    try:
        diagonalize([[0, -1], [1, 0]])
        raise AssertionError("3. Complex eigenvalues: expected ValueError")
    except ValueError as ex:
        ok = "complex" in str(ex).lower()
        assert ok, f"3. Complex eigenvalues: unexpected message: {ex}"
        tag = "OK" if ok else "OK (unexpected message)"
        print(f"3. Complex eigenvalues: {tag} — {ex}")


test_diagonalize_errors()

--- Testing diagonalize error handling ---
1. Non-square: OK — ValueError: Matrix must be square
2. Not diagonalizable: OK — A is not diagonalizable
3. Complex eigenvalues: OK — Matrix has complex eigenvalues; cannot diagonalize over the real numbers
